# AZEquiScope Data Import Pipeline

Comprehensive API-based data import for Arizona healthcare equity analysis.
Combines Census, CDC PLACES, and NPI Registry data for Maricopa County ZCTAs.

## Data Sources
1. **Census API** - Demographics, income, insurance coverage
2. **CDC PLACES API** - Chronic disease prevalence and health indicators
3. **NPI Registry API** - Healthcare provider locations and practice information

## Output
Single comprehensive raw dataset for Maricopa County ZCTAs

## 1. Setup & Configuration

In [3]:
import os
import io
import time
import requests
import pandas as pd
from typing import Optional, Set

print("✓ Imports loaded")

✓ Imports loaded


In [4]:
# Configuration
CENSUS_YEAR = 2023
MARICOPA_FIPS = "04013"  # Arizona state 04 + Maricopa county 013

# API endpoints - all data source endpoints centralized here
CENSUS_ENDPOINT = f"https://api.census.gov/data/{CENSUS_YEAR}/acs/acs5"
SUBJECT_ENDPOINT = f"https://api.census.gov/data/{CENSUS_YEAR}/acs/acs5/subject"
CDC_PLACES_ENDPOINT = "https://chronicdata.cdc.gov/resource/c7b2-4ecy.json"  # ZCTA-level 2023 data JSON API
NPI_REGISTRY_ENDPOINT = "https://npiregistry.cms.hhs.gov/api"  # NPI Registry API 

# Geographic crosswalk
ZCTA2COUNTY_URL = (
    "https://www2.census.gov/geo/docs/maps-data/data/rel2020/zcta520/"
    "tab20_zcta520_county20_natl.txt"
)

print(f"✓ Configuration loaded - Target: Maricopa County, AZ")

✓ Configuration loaded - Target: Maricopa County, AZ


### Data Variable Definitions

In [5]:
# ============================================================================
# DATA SOURCE VARIABLE DEFINITIONS
# ============================================================================

# CENSUS - Demographics & Socioeconomic Data
CENSUS_VARIABLES = {
    "NAME": "Geographic area name (ZCTA identifier)",
    "B01001_001E": "Total population",
    "B19013_001E": "Median household income (2023 inflation-adjusted dollars)",
    "S2701_C03_001E": "Percent insured (civilian noninstitutionalized population)",
    "S2701_C05_001E": "Percent uninsured (civilian noninstitutionalized population)"
}

# CDC PLACES - Chronic Disease & Health Indicators (Crude Prevalence %)
CDC_VARIABLES = {
    # Population Base
    "TotalPopulation": "Total population for ZCTA (CDC estimate)",
    
    # Prevention
    "ACCESS2_CrudePrev": "Current lack of health insurance among adults aged 18–64 years",
    "CHECKUP_CrudePrev": "Visits to doctor for routine checkup within the past year among adults aged ≥18 years",
    "DENTAL_CrudePrev": "Visits to dentist or dental clinic within the past year among adults aged ≥18 years",
    
    # Health Outcomes
    "ARTHRITIS_CrudePrev": "Arthritis among adults aged ≥18 years",
    "BPHIGH_CrudePrev": "High blood pressure among adults aged ≥18 years",
    "CANCER_CrudePrev": "Cancer (excluding skin cancer) among adults aged ≥18 years",
    "CASTHMA_CrudePrev": "Current asthma among adults aged ≥18 years",
    "CHD_CrudePrev": "Coronary heart disease among adults aged ≥18 years",
    "COPD_CrudePrev": "Chronic obstructive pulmonary disease among adults aged ≥18 years",
    "DEPRESSION_CrudePrev": "Depression among adults aged ≥18 years",
    "DIABETES_CrudePrev": "Diabetes among adults aged ≥18 years",
    "HIGHCHOL_CrudePrev": "High cholesterol among adults aged ≥18 years",
    "KIDNEY_CrudePrev": "Chronic kidney disease among adults aged ≥18 years",
    "OBESITY_CrudePrev": "Obesity among adults aged ≥18 years",
    "STROKE_CrudePrev": "Stroke among adults aged ≥18 years",
    
    # Health Status & Quality of Life
    "GHLTH_CrudePrev": "General health among adults aged ≥18 years",
    "MHLTH_CrudePrev": "Mental health not good for ≥14 days among adults aged ≥18 years",
    "PHLTH_CrudePrev": "Physical health not good for ≥14 days among adults aged ≥18 years",
    
    # Health-Related Social Needs
    "FOODINSEC_CrudePrev": "Food insecurity among adults aged ≥18 years",
    "FOODSTAMP_CrudePrev": "Current receipt of food stamps/SNAP benefits among adults aged ≥18 years",
    "HOUSINSECU_CrudePrev": "Housing insecurity among adults aged ≥18 years",
    "LACKTRPT_CrudePrev": "Transportation barriers among adults aged ≥18 years",
    "SHUTULITY_CrudePrev": "Utility shut-offs among adults aged ≥18 years"
}

# NPI REGISTRY - Healthcare Provider Information & Practice Locations
NPI_VARIABLES = {
    "npi": "National Provider Identifier (unique provider ID)",
    "name_first": "Provider first name (individual providers)",
    "name_last": "Provider last name (individual providers)",
    "organization_name": "Organization name (organizational providers)",
    "enumeration_type": "Provider type (NPI-1=Individual, NPI-2=Organization)",
    "practice_address": "Primary practice location street address",
    "practice_city": "Primary practice location city",
    "practice_state": "Primary practice location state (common state column for filtering)",
    "practice_postal_code": "Primary practice location ZIP/postal code (maps to ZCTA okay for this)",
    "practice_phone": "Primary practice location phone number",
    "address_purpose": "Address type (LOCATION=primary practice, PRIMARY=mailing)"
}

# Helper function for adding prefixes to dataframe columns
def add_column_prefix(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """Add prefix to all columns except 'zcta'."""
    df = df.copy()
    df.columns = [f"{prefix}{col}" if col != "zcta" else col for col in df.columns]
    return df

print("✓ Variable definitions and helper functions loaded")

✓ Variable definitions and helper functions loaded


## 2. Geographic Scope - Maricopa County ZCTAs

In [6]:
def fetch_zcta_county_crosswalk() -> Set[str]:
    """
    Fetch ZCTA to County crosswalk and filter to Maricopa County.
    
    Returns:
        Set[str]: Set of ZCTA codes that overlap with Maricopa County
    """
    print("Fetching ZCTA→County crosswalk...")
    
    r = requests.get(ZCTA2COUNTY_URL, timeout=60)
    r.raise_for_status()
    
    # Read pipe-delimited crosswalk file
    cross = pd.read_csv(io.StringIO(r.text), sep="|", dtype=str)
    
    # Keep only ZCTA and county FIPS columns
    keep = cross[["GEOID_ZCTA5_20", "GEOID_COUNTY_20"]].dropna()
    keep = keep.rename(columns={"GEOID_ZCTA5_20": "zcta", "GEOID_COUNTY_20": "county_fips"})
    
    # Filter to Maricopa County
    maricopa_zctas = keep.loc[keep["county_fips"] == MARICOPA_FIPS, "zcta"].drop_duplicates()
    
    print(f"Found {len(maricopa_zctas)} ZCTAs in Maricopa County")
    return set(maricopa_zctas.tolist())

# Get Maricopa County ZCTAs (establishes our geographic scope)
maricopa_zctas = fetch_zcta_county_crosswalk()
print(f"✓ Maricopa County ZCTAs: {len(maricopa_zctas)}")
None  # explicitly suppress display hook

Fetching ZCTA→County crosswalk...
Found 140 ZCTAs in Maricopa County
✓ Maricopa County ZCTAs: 140


## 3. Census API - Demographics & Socioeconomic Data

In [7]:
def fetch_census_data(api_key: Optional[str] = None) -> pd.DataFrame:
    """
    Fetch Census data for all ZCTAs nationwide using standardized API pattern.
    
    Args:
        api_key: Optional Census API key for higher rate limits
        
    Returns:
        pd.DataFrame: Census data with population, income, and insurance coverage
    """
    print(f"Fetching Census {CENSUS_YEAR} data via API...")
    
    # Separate variables by endpoint (regular vs subject tables)
    regular_vars = ["NAME", "B01001_001E", "B19013_001E"]
    subject_vars = ["NAME", "S2701_C03_001E", "S2701_C05_001E"]
    
    try:
        # Fetch regular Census variables
        print(f"Census API call 1/2: Regular variables from {CENSUS_ENDPOINT}")
        params_regular = {
            "get": ",".join(regular_vars),
            "for": "zip code tabulation area:*"
        }
        if api_key:
            params_regular["key"] = api_key

        print("   → Making API request...")
        response1 = requests.get(CENSUS_ENDPOINT, params=params_regular, timeout=30)
        response1.raise_for_status()
        data1 = response1.json()
        df_regular = pd.DataFrame(data1[1:], columns=data1[0])
        
        print(f"✅ Regular Census variables: {len(df_regular)} ZCTAs nationwide")
        
        # Fetch subject table variables
        print(f"Census API call 2/2: Subject variables from {SUBJECT_ENDPOINT}")
        params_subject = {
            "get": ",".join(subject_vars),
            "for": "zip code tabulation area:*"
        }
        if api_key:
            params_subject["key"] = api_key

        print("   → Making API request...")
        response2 = requests.get(SUBJECT_ENDPOINT, params=params_subject, timeout=30)
        response2.raise_for_status()
        data2 = response2.json()
        df_subject = pd.DataFrame(data2[1:], columns=data2[0])
        
        print(f"✅ Subject Census variables: {len(df_subject)} ZCTAs nationwide")
        
        # Merge the dataframes on ZCTA
        df_census = df_regular.merge(df_subject, on=["NAME", "zip code tabulation area"], how="inner")

        # Rename geography column only (no data cleaning here)
        df_census = df_census.rename(columns={"zip code tabulation area": "zcta"})
        
        print(f"✅ Successfully merged Census data for {len(df_census)} ZCTAs nationwide")
        
        return df_census
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Census API request failed: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"❌ Error processing Census API response: {e}")
        return pd.DataFrame()

# Fetch Census data
census_api_key = os.environ.get("CENSUS_API_KEY", None)
census_data = fetch_census_data(census_api_key)

if len(census_data) > 0:
    print(f"✓ Census data downloaded: {len(census_data)} ZCTAs")
    
    # Filter Census data to Maricopa County
    census_maricopa = census_data[census_data["zcta"].isin(maricopa_zctas)].copy()
    print(f"✓ Maricopa ONLY Census data: {len(census_maricopa)} ZCTAs, {len(census_maricopa.columns)} variables")

    # Inspect Census data
    print("\nCensus data preview:")
    display(census_maricopa.head(3))
else:
    print("✗ No Census data retrieved")

None  # explicitly suppress display hook

Fetching Census 2023 data via API...
Census API call 1/2: Regular variables from https://api.census.gov/data/2023/acs/acs5
   → Making API request...
✅ Regular Census variables: 33772 ZCTAs nationwide
Census API call 2/2: Subject variables from https://api.census.gov/data/2023/acs/acs5/subject
   → Making API request...
✅ Subject Census variables: 33772 ZCTAs nationwide
✅ Successfully merged Census data for 33772 ZCTAs nationwide
✓ Census data downloaded: 33772 ZCTAs
✓ Maricopa ONLY Census data: 140 ZCTAs, 6 variables

Census data preview:


,NAME,B01001_001E,B19013_001E,zcta,S2701_C03_001E,S2701_C05_001E
29626,ZCTA5 85003,10155,56672,85003,90.6,9.4
29627,ZCTA5 85004,11178,71250,85004,87.3,12.7
29628,ZCTA5 85006,22081,60742,85006,76.0,24.0


## 4. CDC PLACES API - Health Indicators

In [8]:
def fetch_cdc_places_api(maricopa_zctas: Set[str], year: int = 2023) -> pd.DataFrame:
    """
    Fetch CDC PLACES data for Maricopa County ZCTAs.
    Only retrieves the specific variables defined in CDC_VARIABLES.
    
    Args:
        maricopa_zctas: Set of Maricopa County ZCTA codes to filter for
        year: Data year (default: 2023)
        
    Returns:
        pd.DataFrame: CDC PLACES health indicator data for Maricopa County ZCTAs
    """
    print(f"Fetching CDC PLACES {year} data via API...")
    
    # Build ZCTA filter for Maricopa County
    zcta_list = list(maricopa_zctas)
    zcta_filter = " OR ".join([f"zcta5='{zcta}'" for zcta in zcta_list])
    
    try:
        print("CDC PLACES API call: Maricopa County ZCTAs")
        params = {
            "$where": zcta_filter,
            "$limit": 5000
        }
        
        print("   → Making API request...")
        response = requests.get(CDC_PLACES_ENDPOINT, params=params, timeout=60)
        response.raise_for_status()
        data = response.json()
        df_cdc = pd.DataFrame(data)
        
        print(f"✅ CDC PLACES data: {len(df_cdc)} ZCTA records for Maricopa County")
        
        # Rename zcta5 to zcta for consistency with other datasets
        df_cdc = df_cdc.rename(columns={'zcta5': 'zcta'})
        
        # Filter to target variables (case-insensitive matching)
        available_cols_lower = {col.lower(): col for col in df_cdc.columns}
        keep_cols = ['zcta']
        
        for var in CDC_VARIABLES.keys():
            if var in df_cdc.columns:
                keep_cols.append(var)
            elif var.lower() in available_cols_lower:
                keep_cols.append(available_cols_lower[var.lower()])
        
        # Keep only target columns
        final_cols = [col for col in keep_cols if col in df_cdc.columns]
        df_cdc = df_cdc[final_cols].copy()
        
        print(f"✅ Filtered to {len(final_cols)} columns for {len(df_cdc)} ZCTAs")
        
        return df_cdc
        
    except requests.exceptions.RequestException as e:
        print(f"❌ CDC PLACES API request failed: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"❌ Error processing CDC PLACES API response: {e}")
        return pd.DataFrame()

# Fetch CDC PLACES data
cdc_data = fetch_cdc_places_api(maricopa_zctas, 2023)

if len(cdc_data) > 0:
    print(f"✓ CDC PLACES data: {len(cdc_data)} records")
    
    if 'zcta' in cdc_data.columns:
        unique_zctas = cdc_data['zcta'].nunique()
        print(f"✓ CDC data covers {unique_zctas}/{len(maricopa_zctas)} Maricopa County ZCTAs")
    
    display(cdc_data.head(3))
else:
    print("✗ No CDC PLACES data retrieved")

None  # explicitly suppress display hook

Fetching CDC PLACES 2023 data via API...
CDC PLACES API call: Maricopa County ZCTAs
   → Making API request...
✅ CDC PLACES data: 136 ZCTA records for Maricopa County
✅ Filtered to 20 columns for 136 ZCTAs
✓ CDC PLACES data: 136 records
✓ CDC data covers 136/140 Maricopa County ZCTAs


,zcta,totalpopulation,access2_crudeprev,checkup_crudeprev,dental_crudeprev,arthritis_crudeprev,bphigh_crudeprev,cancer_crudeprev,casthma_crudeprev,chd_crudeprev,copd_crudeprev,depression_crudeprev,diabetes_crudeprev,highchol_crudeprev,kidney_crudeprev,obesity_crudeprev,stroke_crudeprev,ghlth_crudeprev,mhlth_crudeprev,phlth_crudeprev
0,85003,9369,16.6,65.0,57.4,17.5,25.4,4.5,10.0,4.7,5.1,18.8,9.7,30.0,2.7,31.8,2.5,17.2,18.1,10.9
1,85004,4965,16.4,65.5,56.1,17.3,24.8,4.6,11.0,4.6,5.3,21.3,9.0,29.5,2.7,31.3,2.4,17.3,20.8,11.0
2,85006,25742,28.5,64.4,45.7,19.2,28.2,4.3,11.2,5.6,6.7,20.8,13.0,31.9,3.4,37.6,3.1,25.7,20.9,14.8


## 5. NPI Registry API - Healthcare Providers

In [9]:
def fetch_npi_providers_api(maricopa_zctas: Set[str], state: str = "AZ") -> pd.DataFrame:
    """
    Fetch healthcare provider data from NPI Registry API by querying each ZCTA postal code.
    
    Note: Unlike CDC PLACES (which returns ZCTA-level data in one call), NPI API requires
    querying each postal code individually since it doesn't support bulk ZCTA filtering.
    This results in 140 individual API calls for Maricopa County.
    
    New Implementation Details:
    - Only collects LOCATION addresses (where patients receive care)
    - Handles providers with multiple practice locations correctly
    - Removes duplicate provider-location combinations
    - Each record represents one provider at one practice location
    
    Args:
        maricopa_zctas: Set of Maricopa County ZCTA codes to query
        state: State abbreviation (default: "AZ" for Arizona)  
        
    Returns:
        pd.DataFrame: Healthcare provider data from NPI Registry (provider-location records)
    """
    postal_codes = sorted(maricopa_zctas)
    print(f"Fetching NPI provider data for {len(postal_codes)} Maricopa County ZCTAs")
    print(f"This will take ~{len(postal_codes) * 0.3 / 60:.1f} minutes with rate limiting...")
    
    all_providers = []
    successful = 0
    failed = 0
    
    try:
        for idx, postal_code in enumerate(postal_codes, 1):
            # Progress updates every 20 queries
            if idx % 20 == 0:
                print(f"Progress: {idx}/{len(postal_codes)} queries, {len(all_providers)} providers found")
            
            try:
                params = {
                    "version": "2.1",
                    "postal_code": postal_code,
                    "state": state,
                    "limit": 200
                }
                
                response = requests.get(NPI_REGISTRY_ENDPOINT, params=params, timeout=30)
                response.raise_for_status()
                results = response.json().get("results", [])
                
                if results:
                    successful += 1
                    
                    for provider in results:
                        basic = provider.get("basic", {})
                        addresses = provider.get("addresses", [])
                        
                        # Get ALL LOCATION addresses (providers may have multiple practice locations)
                        location_addresses = [
                            addr for addr in addresses 
                            if addr.get("address_purpose") == "LOCATION"
                        ]
                        
                        # Add each LOCATION address as a separate provider record
                        for location_addr in location_addresses:
                            if location_addr:  # Ensure address exists
                                all_providers.append({
                                    "npi": provider.get("number"),
                                    "name_first": basic.get("first_name"),
                                    "name_last": basic.get("last_name"), 
                                    "organization_name": basic.get("organization_name"),
                                    "enumeration_type": provider.get("enumeration_type"),
                                    "practice_address": location_addr.get("address_1"),
                                    "practice_city": location_addr.get("city"),
                                    "practice_state": location_addr.get("state"),
                                    "practice_postal_code": location_addr.get("postal_code"),
                                    "practice_phone": location_addr.get("telephone_number"),
                                    "address_purpose": location_addr.get("address_purpose")
                                })
                
                time.sleep(0.3)  # Rate limiting
                
            except requests.exceptions.RequestException as e:
                failed += 1
                if failed <= 3:
                    print(f"  Failed to query {postal_code}: {e}")
                continue
        
        print(f"\n✅ NPI query complete: {successful}/{len(postal_codes)} successful, {len(all_providers)} provider records")
        
        # Convert to DataFrame and remove duplicates
        df_providers = pd.DataFrame(all_providers)
        
        if len(df_providers) > 0:
            # Add ZCTA column for duplicate handling
            df_providers['zcta'] = df_providers['practice_postal_code'].astype(str).str[:5]
            
            # Remove duplicates based on NPI + ZCTA (same provider in same ZCTA)
            initial_count = len(df_providers)
            df_providers = df_providers.drop_duplicates(
                subset=['npi', 'zcta'], 
                keep='first'
            )
            
            duplicates_removed = initial_count - len(df_providers)
            if duplicates_removed > 0:
                print(f"    Removed {duplicates_removed} duplicate provider-ZCTA records")
            
            print(f"    Final result: {len(df_providers)} unique provider-ZCTA combinations")
            
        return df_providers
        
    except Exception as e:
        print(f"❌ NPI API error: {e}")
        return pd.DataFrame()

# Fetch NPI provider data
npi_data = fetch_npi_providers_api(maricopa_zctas)

if len(npi_data) > 0:
    # Filter to only Maricopa County ZCTAs (removes providers with addresses outside Maricopa)
    # Note: zcta column already created in function
    npi_data_all = npi_data.copy()  # Keep unfiltered for debugging if needed
    npi_data = npi_data[npi_data['zcta'].isin(maricopa_zctas)].copy()
    
    unique_postal = npi_data['practice_postal_code'].nunique()
    unique_zctas = npi_data['zcta'].nunique()
    print(f"✓ Filtered to {len(npi_data)} providers in Maricopa County")
    print(f"  {unique_zctas} ZCTAs with providers, {unique_postal} unique postal codes")
    print(f"  ({len(npi_data_all) - len(npi_data)} providers outside Maricopa excluded)")
    
    # Create provider counts by type per ZCTA
    print("\nCreating provider counts by ZCTA...")
    
    # Individual providers (NPI-1) per ZCTA
    npi1_per_zcta = npi_data[npi_data['enumeration_type'] == 'NPI-1'].groupby("zcta")["npi"].nunique().reset_index()
    npi1_per_zcta.columns = ["zcta", "individual_providers"]
    
    # Organizational providers (NPI-2) per ZCTA  
    npi2_per_zcta = npi_data[npi_data['enumeration_type'] == 'NPI-2'].groupby("zcta")["npi"].nunique().reset_index()
    npi2_per_zcta.columns = ["zcta", "organization_providers"]
    
    # Total providers per ZCTA
    total_per_zcta = npi_data.groupby("zcta")["npi"].nunique().reset_index()
    total_per_zcta.columns = ["zcta", "total_providers"]
    
    # Merge all counts
    provider_counts = total_per_zcta.merge(npi1_per_zcta, on="zcta", how="left")
    provider_counts = provider_counts.merge(npi2_per_zcta, on="zcta", how="left")
    provider_counts = provider_counts.fillna(0).astype({"individual_providers": int, "organization_providers": int})
    
    print(f"✓ Provider counts created for {len(provider_counts)} ZCTAs")
    print("\nProvider summary by type:")
    print(f"  Individual (NPI-1): {provider_counts['individual_providers'].sum():,} total across all ZCTAs")
    print(f"  Organizations (NPI-2): {provider_counts['organization_providers'].sum():,} total across all ZCTAs") 
    print(f"  Combined total: {provider_counts['total_providers'].sum():,} unique providers")
    
    print("\nProvider counts by ZCTA (first 5 rows):")
    display(provider_counts.head())
    
    print("\nNPI data preview:")
    display(npi_data.head(3))
else:
    print("✗ No NPI data retrieved")

None  # explicitly suppress display hook

Fetching NPI provider data for 140 Maricopa County ZCTAs
This will take ~0.7 minutes with rate limiting...
Progress: 20/140 queries, 3800 providers found
Progress: 40/140 queries, 7492 providers found
Progress: 60/140 queries, 11389 providers found
Progress: 80/140 queries, 15254 providers found
Progress: 100/140 queries, 18890 providers found
Progress: 120/140 queries, 21360 providers found
Progress: 140/140 queries, 24814 providers found

✅ NPI query complete: 139/140 successful, 24815 provider records
    Removed 3577 duplicate provider-ZCTA records
    Final result: 21238 unique provider-ZCTA combinations
✓ Filtered to 19710 providers in Maricopa County
  136 ZCTAs with providers, 9937 unique postal codes
  (1528 providers outside Maricopa excluded)

Creating provider counts by ZCTA...
✓ Provider counts created for 136 ZCTAs

Provider summary by type:
  Individual (NPI-1): 12,323 total across all ZCTAs
  Organizations (NPI-2): 7,387 total across all ZCTAs
  Combined total: 19,710 u

,zcta,total_providers,individual_providers,organization_providers
0,85003,137,96,41
1,85004,181,122,59
2,85006,243,179,64
3,85007,167,115,52
4,85008,199,138,61



NPI data preview:


,npi,name_first,name_last,organization_name,enumeration_type,practice_address,practice_city,practice_state,practice_postal_code,practice_phone,address_purpose,zcta
0,1649361783,None,None,"A NEW PERSPECTIVE COUNSELING CENTER, INC.",NPI-2,420 W ROOSEVELT ST,PHOENIX,AZ,850031325,602-264-2893,LOCATION,85003
1,1437378858,None,None,A-MCDOWELL DENTAL,NPI-2,125 WEST MCDOWELL ROAD,PHOENIX,AZ,850031223,602-273-0013,LOCATION,85003
2,1184317042,LINDA,ABEGG,None,NPI-1,2 W VERNON AVE,PHOENIX,AZ,850031039,801-358-1736,LOCATION,85003


## 6. Data Integration & ZCTA-Level Merging

Combine Census, CDC PLACES, and NPI data into a single comprehensive dataset at the ZCTA level.

In [10]:
# Data validation and base dataset creation
print("Data validation:")
print(f"  Census: {len(census_maricopa)} ZCTAs")
print(f"  CDC: {len(cdc_data)} records") 
print(f"  NPI Aggregate: {len(provider_counts)} records")

# Note: Census vs CDC coverage difference is expected
# Census includes ALL ZCTAs in Maricopa County (including low-population/rural areas)
# CDC PLACES only includes ZCTAs with sufficient population for reliable health estimates
# The ~4 ZCTA difference represents areas that are likely rural/industrial with minimal residential population

if len(census_maricopa) == 0:
    raise ValueError("✗ No Census data for Maricopa County")
else:
    print("✓ Validation passed")

# Create base dataset from Census data with standardized column names
merged_data = census_maricopa.copy()

# Standardize Census columns with prefix
merged_data = add_column_prefix(merged_data, "census_")

print(f"✓ Base dataset (Census) created: {len(merged_data)} ZCTAs, {len(merged_data.columns)} columns")

Data validation:
  Census: 140 ZCTAs
  CDC: 136 records
  NPI Aggregate: 136 records
✓ Validation passed
✓ Base dataset (Census) created: 140 ZCTAs, 6 columns


### Merge CDC PLACES data at ZCTA level

In [11]:
# Merge CDC PLACES data at ZCTA level
if len(cdc_data) > 0:
    print(f"Processing CDC PLACES: {len(cdc_data)} records")
    
    # Standardize CDC data to ZCTA level
    cdc_zcta = cdc_data.copy()
    
    # Apply CDC prefix to all columns except zcta
    cdc_zcta = add_column_prefix(cdc_zcta, "cdc_")
    
    # ZCTA-level merge
    merged_data = merged_data.merge(cdc_zcta, on="zcta", how="left")
    unique_cdc_zctas = len(cdc_zcta["zcta"].unique())
    print(f"   ✓ CDC merged: {unique_cdc_zctas} unique ZCTAs")
else:
    print("✗ No CDC PLACES data to merge")

print(f"Dataset after CDC merge: {merged_data.shape}")

Processing CDC PLACES: 136 records
   ✓ CDC merged: 136 unique ZCTAs
Dataset after CDC merge: (140, 25)


### Merge NPI data at ZCTA level

In [12]:
# Merge NPI provider data at ZCTA level
if len(provider_counts) > 0:
    print(f"Merging provider counts: {len(provider_counts)} ZCTAs with provider data")
    
    # Use the provider_counts DataFrame created in the NPI API section
    # Add npi_ prefix to provider count columns for consistency
    npi_zcta = provider_counts.copy()
    npi_zcta = npi_zcta.rename(columns={
        "total_providers": "npi_total_providers",
        "individual_providers": "npi_individual_providers", 
        "organization_providers": "npi_organization_providers"
    })
    
    # ZCTA-level merge with Census+CDC data
    merged_data = merged_data.merge(npi_zcta, on="zcta", how="left")
    print(f"   ✓ Provider counts merged: {len(npi_zcta)} ZCTAs with provider data")
    
    # Display merge results
    zctas_with_providers = merged_data['npi_total_providers'].gt(0).sum()
    total_providers_merged = merged_data['npi_total_providers'].sum()
    print(f"   ✓ {zctas_with_providers}/{len(merged_data)} ZCTAs have provider data")
    print(f"   ✓ {total_providers_merged:,} total providers across all ZCTAs")
else:
    print("✗ No NPI data to merge")

print(f"Dataset after NPI merge: {merged_data.shape}")

# Calculate completeness BEFORE filling missing values (for accurate reporting)
cdc_cols = [c for c in merged_data.columns if c.startswith("cdc_")]
npi_cols = [c for c in merged_data.columns if c.startswith("npi_")]

if cdc_cols:
    cdc_complete_count = merged_data[cdc_cols].notna().any(axis=1).sum()
else:
    cdc_complete_count = 0

if npi_cols:
    npi_complete_count = merged_data[npi_cols].notna().any(axis=1).sum()
else:
    npi_complete_count = 0

# Fill missing values (ZCTAs without providers/health data)
fill_cols = [col for col in merged_data.columns if col.startswith(("npi_", "cdc_"))]
for col in fill_cols:
    merged_data[col] = merged_data[col].fillna(0)

print(f"✓ Filled missing values for {len(fill_cols)} columns")
print(f"Final dataset shape: {merged_data.shape}")

Merging provider counts: 136 ZCTAs with provider data
   ✓ Provider counts merged: 136 ZCTAs with provider data
   ✓ 136/140 ZCTAs have provider data
   ✓ 19,710.0 total providers across all ZCTAs
Dataset after NPI merge: (140, 28)
✓ Filled missing values for 22 columns
Final dataset shape: (140, 28)


In [19]:
# Get all ZCTAs from each dataset
census_zctas = set(census_maricopa['zcta'].unique())
cdc_zctas = set(cdc_data['zcta'].unique()) if len(cdc_data) > 0 else set()
npi_zctas = set(provider_counts['zcta'].unique()) if len(provider_counts) > 0 else set()

print(f"\nZCTA counts by dataset:")
print(f"  Census (all Maricopa): {len(census_zctas)} ZCTAs")
print(f"  CDC PLACES: {len(cdc_zctas)} ZCTAs")
print(f"  NPI Registry: {len(npi_zctas)} ZCTAs with providers")

# Find ZCTAs in Census but NOT in CDC
missing_from_cdc = census_zctas - cdc_zctas
if missing_from_cdc:
    print(f"\n⚠️  ZCTAs in Census but MISSING from CDC PLACES ({len(missing_from_cdc)} ZCTAs):")
    for zcta in sorted(missing_from_cdc):
        # Get Census population for context
        census_pop = merged_data.loc[merged_data['zcta'] == zcta, 'census_B01001_001E'].values
        census_pop_str = f"{int(census_pop[0]):,}" if len(census_pop) > 0 and not pd.isna(census_pop[0]) else "N/A"
        
        # Get CDC population (should be 0 or NaN since ZCTA is missing from CDC)
        cdc_pop = merged_data.loc[merged_data['zcta'] == zcta, 'cdc_totalpopulation'].values
        cdc_pop_str = f"{int(cdc_pop[0]):,}" if len(cdc_pop) > 0 and not pd.isna(cdc_pop[0]) and cdc_pop[0] != 0 else "N/A"
        
        print(f"    {zcta} | Census pop: {census_pop_str} | CDC pop: {cdc_pop_str}")
else:
    print(f"\n✅ All Census ZCTAs have CDC PLACES data") 
    
# Find ZCTAs in Census but NOT in NPI (ZCTAs without providers)
census_not_in_npi = census_zctas - npi_zctas
if census_not_in_npi:
    print(f"\n⚠️  ZCTAs in Census but NOT in NPI Registry ({len(census_not_in_npi)} ZCTAs):")
    for zcta in sorted(census_not_in_npi):
        # Get Census population for context
        census_pop = merged_data.loc[merged_data['zcta'] == zcta, 'census_B01001_001E'].values
        census_pop_str = f"{int(census_pop[0]):,}" if len(census_pop) > 0 and not pd.isna(census_pop[0]) else "N/A"
        print(f"    {zcta} (Census population: {census_pop_str})")
else:
    print(f"\n✅ All Census ZCTAs have provider data")


ZCTA counts by dataset:
  Census (all Maricopa): 140 ZCTAs
  CDC PLACES: 136 ZCTAs
  NPI Registry: 136 ZCTAs with providers

⚠️  ZCTAs in Census but MISSING from CDC PLACES (4 ZCTAs):
    85026 | Census pop: 0 | CDC pop: N/A
    85236 | Census pop: 0 | CDC pop: N/A
    85329 | Census pop: 2,429 | CDC pop: N/A
    85378 | Census pop: 9,401 | CDC pop: N/A

⚠️  ZCTAs in Census but NOT in NPI Registry (4 ZCTAs):
    85026 (Census population: 0)
    85320 (Census population: 315)
    85322 (Census population: 493)
    85545 (Census population: 669)


## 7. Final Dataset Export 

In [ ]:
# Simple export and summary
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

# Reorder columns to put zcta first
cols = merged_data.columns.tolist()
if 'zcta' in cols:
    cols.remove('zcta')
    cols = ['zcta'] + cols
    merged_data = merged_data[cols]

# Save the dataset
merged_file = f"{output_dir}/maricopa_healthcare_raw_data.csv"
merged_data.to_csv(merged_file, index=False)

# Simple summary output
print("=" * 50)
print("DATA EXPORT COMPLETE")
print("=" * 50)
print(f"Dataset saved: {merged_file}")
print(f"Total ZCTAs: {len(merged_data)}")
print(f"Total columns: {len(merged_data.columns)}")
print(f"✓ Column order: zcta first, followed by {len(cols)-1} data columns")

DATA EXPORT COMPLETE
Dataset saved: data/maricopa_healthcare_raw_data.csv
Total ZCTAs: 140
Total columns: 28
✓ Column order: zcta first, followed by 27 data columns
